This code exports bcCBI datasets to a Google Drive for three different fire perimeter sets: Welty & Jeffries combined wildland fire perimeters, FIRED, and WUMI.

In [65]:
import os
import re
from datetime import datetime
import pandas as pd

import ee
import geemap  # (not strictly needed below, but fine to keep)


#####################################
#    SET ALL PARAMETERS #############
#####################################

PROJECT_DIR = "projects/ee-tymc5571-cbi-module/assets"
GEE_PROJECT = "ee-tymc5571-cbi-module"
DRIVE_FOLDER      = "GEE_Exports"
BCCBI_ASSET_STRING = "west_wall_to_wall_bccbi"
CRS = "EPSG:5070"  # export CRS
DIR_DERIVED = "C:/Users/tymc5571/dev/cbi-module/data/derived/"
YEARS = None #If not none, will only operate on these years
POLYSETS = ["WELTY", "FIREDv2"] #If not none, will only operate on these polygon sets; 'name'as in registry

# ee.Authenticate()
ee.Authenticate(
    auth_mode='notebook',
    scopes=[
        'https://www.googleapis.com/auth/earthengine',
        'https://www.googleapis.com/auth/devstorage.read_write',
        'https://www.googleapis.com/auth/drive'
    ]
)
ee.Initialize(project=GEE_PROJECT)

# Output region (FeatureCollection → Geometry)
states = ee.FeatureCollection("TIGER/2018/States")
western_states = ['WA','OR','CA','ID','NV','MT','WY','UT','CO','AZ','NM']
OUTPUT_REGION_FC = states.filter(ee.Filter.inList('STUSPS', western_states))
OUTPUT_REGION = OUTPUT_REGION_FC.geometry()

# Polygon sets
WUMI_circles  = ee.FeatureCollection(f"{PROJECT_DIR}/WUMI2024a_main_fires_unified_simple_with_circles")
WUMI_nocirc   = ee.FeatureCollection(f"{PROJECT_DIR}/WUMI2024a_main_fires_unified_simple_no_circles")
FIREDv2       = ee.FeatureCollection(f"{PROJECT_DIR}/firedv2_west")
WELTY         = ee.FeatureCollection(f"{PROJECT_DIR}/welty_wildfire_west_1984_2020")

registry = [
    {"name": "WUMI circles",    "fc": WUMI_circles, "asset_slug": "wumi_west_circles",   "year_attr": "fire_year"},
    {"name": "WUMI no-circles", "fc": WUMI_nocirc,  "asset_slug": "wumi_west_nocircles", "year_attr": "fire_year"},
    {"name": "FIREDv2",         "fc": FIREDv2,      "asset_slug": "firedv2_west",        "year_attr": "ig_year"},
    {"name": "WELTY",           "fc": WELTY,        "asset_slug": "welty_west",          "year_attr": "Fire_Year"},
]

if POLYSETS is not None:
    registry = [r for r in registry if r["name"] in POLYSETS]

In [66]:
# TESTING

# TEST_YEARS = [2010]
# registry = [
#     {"name": "FIREDv2",         "fc": FIREDv2,      "asset_slug": "firedv2_west",        "year_attr": "ig_year"},
#     {"name": "WELTY",           "fc": WELTY,        "asset_slug": "welty_west",          "year_attr": "Fire_Year"},
# ]

# counties = ee.FeatureCollection("TIGER/2018/Counties")
# boulder_co = counties.filter(ee.Filter.eq('GEOID', '08013')).first()
# OUTPUT_REGION = ee.Feature(boulder_co).geometry()  # geometry of Boulder County


In [67]:
def list_assets(parent: str) -> list[str]:
    """Recursively list asset IDs under an Earth Engine folder/collection."""
    assets = []
    def _list(parent_id):
        response = ee.data.listAssets({'parent': parent_id})
        for item in response.get('assets', []):
            asset_id = item.get('name')
            asset_type = item.get('type')
            if asset_type in ('FOLDER', 'IMAGE_COLLECTION'):
                _list(asset_id)
            else:
                assets.append(asset_id)
    _list(parent)
    return assets

# Gather bc-cbi image assets and wrap as an ImageCollection
dir_assets = list_assets(PROJECT_DIR)
filtered_assets = [a for a in dir_assets if BCCBI_ASSET_STRING in a]
print("Wall-to-wall BC-CBI available:", len(filtered_assets))
bc_cbi_ic = ee.ImageCollection.fromImages([ee.Image(a) for a in filtered_assets])


Wall-to-wall BC-CBI available: 40


In [68]:
# ---------------------------
# Helpers for per-year export
# ---------------------------

def get_unique_years(polygons: ee.FeatureCollection, fire_year_attr: str) -> list[int]:
    """
    Return a Python list of unique years (ints) present in the polygon set.
    (Client-side list is necessary to schedule export tasks.)
    """
    years = (polygons
             .aggregate_array(fire_year_attr)
             .distinct()
             .sort())
    # Safe to fetch: typically a few dozen years at most
    py_years = years.getInfo()
    # Normalize to ints and filter out nulls/NaNs
    clean = []
    for y in py_years:
        try:
            if y is not None:
                clean.append(int(float(y)))
        except Exception:
            pass
    return sorted(set(clean))

def build_year_image(year: int,
                     polygons: ee.FeatureCollection,
                     fire_year_attr: str) -> ee.Image:
    """
    Server-side: mosaic the CBI_bc tiles for `year`, mask to that year's polygons,
    and return a single-band image named 'CBI_bc' (no stacking).
    """
    y = ee.Number(year)
    # Polygons (fires) for that year
    year_fires = polygons.filter(ee.Filter.eq(fire_year_attr, y))

    # Mask: any pixel covered by ≥1 polygon for that year
    fire_mask = year_fires.reduceToImage(
        properties=[fire_year_attr],
        reducer=ee.Reducer.count()
    ).gt(0)

    # Year image: mosaic all tiles tagged with that 'year'
    year_img = (bc_cbi_ic
                .filter(ee.Filter.eq('year', y))
                .select('CBI_bc')
                .mosaic()
                .toFloat()
                .updateMask(fire_mask)
                .rename('CBI_bc'))

    # (Optional) carry a property for traceability
    year_img = year_img.set({
        'year': y,
        'masked_by': fire_year_attr
    })
    return year_img


def exports(image: ee.Image,
                region_geom: ee.Geometry,
                asset_folder_base: str,
                base_slug: str,
                *,
                suffix: str = "",
                description_prefix: str = "bccbi",
                drive_folder: str = "GEE_Exports",
                scale: int = 30,
                crs: str = "EPSG:5070",
                maxPixels: float = 1e13):
    """
    Export `image` both to Asset and Drive with a consistent naming scheme.
    `suffix` lets you embed the year (e.g., "_2005").
    """
    # e.g., bccbi_wumi_west_circles_y2005
    desc = f"{description_prefix}_{base_slug}{suffix}"

    # Assets: full assetId (no extension)
    asset_id = f"{asset_folder_base}/{desc}"

    # Export to Asset
    task_asset = ee.batch.Export.image.toAsset(
        image=image,
        description=desc + "_toAsset",
        assetId=asset_id,
        region=region_geom,
        scale=scale,
        crs=crs,
        maxPixels=maxPixels
    )
    task_asset.start()

    # Export to Drive
    task_drive = ee.batch.Export.image.toDrive(
        image=image,
        description=desc + "_toDrive",
        folder=drive_folder,
        fileNamePrefix=desc,
        region=region_geom,
        scale=scale,
        crs=crs,
        maxPixels=maxPixels
    )
    task_drive.start()

    return {"asset_task": task_asset.id, "drive_task": task_drive.id}


In [ ]:
    

# ------------------------------------
# Iterate registry → years → export
# ------------------------------------

def has_bccbi_for_year(year: int, ic: ee.ImageCollection) -> bool:
    """
    Returns True if bc_cbi_ic has imagery tagged with 'year' == year.
    Uses a lightweight client-side count check.
    """
    count = ic.filter(ee.Filter.eq('year', year)).size().getInfo()
    return count > 0

export_tasks = []

for item in registry:
    fc         = item["fc"]
    year_attr  = item["year_attr"]
    asset_slug = item["asset_slug"]

    export_manifest = []

    years = get_unique_years(fc, year_attr)
    print(f"{item['name']}: found {len(years)} years → {years[:8]}{'...' if len(years) > 8 else ''}")

    if YEARS is not None:
        years = YEARS

    for y in years:

        if not has_bccbi_for_year(y, bc_cbi_ic):
            print(f"Skipping {item['name']} {y}: no bc_cbi imagery available.")
            continue
    
        yr_img = build_year_image(y, fc, year_attr)
        tasks = exports(
            image=yr_img,
            region_geom=OUTPUT_REGION,
            asset_folder_base=PROJECT_DIR,
            base_slug=asset_slug,
            suffix=f"_{y}",
            description_prefix="bccbi",
            drive_folder=DRIVE_FOLDER,
            scale=30,
            crs=CRS,
            maxPixels=1e13
        )
        export_tasks.append({"name": item["name"], "year": y, "tasks": tasks})

        export_manifest.append({
            "id": f"bccbi_{asset_slug}_{y}"
        })

    pd.DataFrame(export_manifest).to_csv(f"{DIR_DERIVED}/{asset_slug}_manifest.csv", index=False)

print("Submitted tasks:", len(export_tasks))

print("Wrote export_manifest.csv")

FIREDv2: found 26 years → [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007]...
Skipping FIREDv2 2025: no bc_cbi imagery available.
WELTY: found 37 years → [1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991]...
Skipping WELTY 1984: no bc_cbi imagery available.
Submitted tasks: 0
Wrote export_manifest.csv


In [70]:
tasks = ee.data.getTaskList()
for t in tasks:
    print(f"[{t['state']:<10}] {t.get('task_type'):<14} {t.get('description')}  id={t.get('id')}")

[COMPLETED ] EXPORT_IMAGE   bccbi_wumi_west_nocircles_2024_toDrive  id=QQFOXOO4QZEHPUCRILUOLWT7
[FAILED    ] EXPORT_IMAGE   bccbi_wumi_west_nocircles_2024_toAsset  id=NAYICHHUO2VZBO7SS5T2NTTB
[FAILED    ] EXPORT_IMAGE   bccbi_welty_west_2020_toDrive  id=3DUB35BMPZAEXCIIQWV24RC3
[COMPLETED ] EXPORT_IMAGE   bccbi_welty_west_2020_toAsset  id=4M2CO7RPM6F3RLBI6XVSXCBI
[FAILED    ] EXPORT_IMAGE   bccbi_welty_west_2019_toDrive  id=7ZCLNMYHBMDBWKSS5WEHVFIS
[COMPLETED ] EXPORT_IMAGE   bccbi_welty_west_2019_toAsset  id=ITJ3ZHIFBWRGJ2VDPEWXLRU7
[FAILED    ] EXPORT_IMAGE   bccbi_welty_west_2018_toDrive  id=6WSJKNZNRLVTFGZBNBREYBA6
[COMPLETED ] EXPORT_IMAGE   bccbi_welty_west_2018_toAsset  id=6RACDBKW6ZJ4JLDV7V2JTIC5
[FAILED    ] EXPORT_IMAGE   bccbi_welty_west_2017_toDrive  id=L2O6GX2JZJTIGIZIKCLKAO5M
[COMPLETED ] EXPORT_IMAGE   bccbi_welty_west_2017_toAsset  id=WQTUNLWGADZYKNULTWIYUCNS
[FAILED    ] EXPORT_IMAGE   bccbi_welty_west_2016_toDrive  id=3HNBR3MEZPHHGRYPZDDMVZXA
[COMPLETED ] EXPORT_IMAGE